# ECH 405 – Week Task: Exploratory Data Analysis (EDA)
**Student:** Prabesh  
**Project:** AI-Based Personalized Diet Recommendation & Nutrition Management System  
**Dataset:** Food Nutrition Dataset (6 CSV groups)  
**Date:** September 2026

---

## 0. Imports & Setup

In [ ]:
import os
import glob
import warnings
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False
})
sns.set_palette('husl')
random.seed(42)
np.random.seed(42)

# Adjust path if running from a different working directory
DATASET_DIR = 'food_dataset'
print('Dataset directory:', os.path.abspath(DATASET_DIR))

---
## Part A – Section 1: Dataset Card

| Field | Details |
|---|---|
| **Source** | [Kaggle – Food Nutritional Values Dataset](https://www.kaggle.com/datasets/utsavdey1410/food-nutrition) |
| **Licence** | CC0: Public Domain (no restrictions) |
| **Size on disk** | ~577 KB across 6 CSV files |
| **Total samples** | **3,295 food items** (551 + 319 + 571 + 232 + 722 + 900 rows across GROUP1–GROUP6) |
| **Features per row** | 37 columns: food name + Caloric Value + 14 macro/micro-nutrients + 15 vitamins/minerals + Nutrition Density + index cols |
| **Collection method** | Compiled from USDA FoodData Central & regional nutrition databases; standardised to per-100 g serving |
| **Known limitations** | (1) All values are **per 100 g** — does not reflect real serving sizes. (2) Some regional/processed foods (e.g. `requeijao cremoso light catupiry`) have **zero-filled** micro-nutrient columns because data was unavailable. (3) No temporal information — cannot derive meal sequence patterns from this dataset directly; sequences must come from logged `MealLog` entries. (4) The 6 groups have no explicit food-category labels; food group membership must be inferred by name or cluster. |

### LSTM Context
The food dataset is the **item look-up catalogue** for the LSTM model.  
The LSTM learns from **sequences of `MealLog` vectors** `[calories, protein, carbohydrates, fat, dietary_fiber]` stored in the app database; it uses this catalogue only when decoding the predicted nutrient vector back into concrete food recommendations.

---
## Part A – Section 2: Load Data & Basic Structure

In [ ]:
# Load all 6 CSV files
csv_paths = sorted(glob.glob(os.path.join(DATASET_DIR, 'FOOD-DATA-GROUP*.csv')))
print(f'Found {len(csv_paths)} CSV files:')

dfs = []
for path in csv_paths:
    df_tmp = pd.read_csv(path)
    df_tmp['source_file'] = os.path.basename(path)
    size_kb = os.path.getsize(path) / 1024
    print(f'  {os.path.basename(path):30s}  rows={len(df_tmp):>4d}  cols={len(df_tmp.columns):>2d}  ({size_kb:.1f} KB)')
    dfs.append(df_tmp)

df = pd.concat(dfs, ignore_index=True)
print(f'\nCombined dataset shape: {df.shape}')

In [ ]:
# Clean up redundant index columns
drop_cols = [c for c in df.columns if c.startswith('Unnamed')]
df.drop(columns=drop_cols, errors='ignore', inplace=True)

# Standardise column names for LSTM features
df.rename(columns={
    'Caloric Value': 'calories',
    'Fat': 'fat',
    'Carbohydrates': 'carbohydrates',
    'Protein': 'protein',
    'Dietary Fiber': 'dietary_fiber',
    'Saturated Fats': 'saturated_fats',
    'Monounsaturated Fats': 'monounsaturated_fats',
    'Polyunsaturated Fats': 'polyunsaturated_fats',
    'Sugars': 'sugars',
    'Cholesterol': 'cholesterol',
    'Sodium': 'sodium',
    'Water': 'water',
    'Nutrition Density': 'nutrition_density'
}, inplace=True)

print('Shape after cleaning:', df.shape)
print('\nColumn dtypes:')
print(df.dtypes.to_string())

In [ ]:
# 5 random samples
LSTM_FEATURES = ['calories', 'protein', 'carbohydrates', 'fat', 'dietary_fiber']
display_cols = ['food', 'source_file'] + LSTM_FEATURES + ['nutrition_density']
print('5 Random Samples (LSTM feature columns highlighted):')
df[display_cols].sample(5, random_state=42)

---
## Part A – Section 3: Missing, Duplicate & Corrupt Entries

In [ ]:
# 3.1  Missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
miss_df = pd.DataFrame({'missing_count': missing, 'missing_%': missing_pct})
miss_df = miss_df[miss_df['missing_count'] > 0].sort_values('missing_count', ascending=False)

if miss_df.empty:
    print('No NaN missing values detected.')
else:
    print('Columns with missing values:')
    print(miss_df.to_string())

In [ ]:
# 3.2  Duplicate rows
full_dups = df.duplicated().sum()
name_dups = df.duplicated(subset=['food']).sum()
print(f'Fully duplicate rows     : {full_dups}')
print(f'Duplicate food names     : {name_dups}')
if name_dups > 0:
    print('\nSome duplicated food names (first 10):')
    dup_names = df[df.duplicated(subset=['food'], keep=False)].sort_values('food')
    print(dup_names[['food', 'source_file'] + LSTM_FEATURES].head(10).to_string(index=False))

In [ ]:
# 3.3  Corrupt / suspicious entries
# Rule 1: Zero-row - all 5 LSTM features are exactly zero
zero_rows = (df[LSTM_FEATURES] == 0).all(axis=1)
print(f'Rows where ALL 5 LSTM features = 0: {zero_rows.sum()}')

# Rule 2: Negative values (physically impossible)
neg_rows = (df[LSTM_FEATURES] < 0).any(axis=1)
print(f'Rows with any negative LSTM feature value: {neg_rows.sum()}')

# Rule 3: Extreme outliers - calories > 1200 kcal per 100g
extreme_cal = df[df['calories'] > 1200]
print(f'Rows with calories > 1200 kcal/100g: {len(extreme_cal)}')
if len(extreme_cal) > 0:
    print(extreme_cal[['food', 'calories'] + LSTM_FEATURES].to_string(index=False))

In [ ]:
# 3.4  Remediation plan
print("""
REMEDIATION PLAN
================
1. MISSING VALUES:
   No NaN values detected; however many micro-nutrient cells contain 0.0
   as a proxy for 'not measured'. For LSTM features (calories, protein,
   carbs, fat, dietary_fiber) this is an actual data quality issue.
   Action: Keep zero-filled rows in the LOOKUP catalogue; exclude them
   from LSTM training sequences where the food was logged.

2. DUPLICATE FOOD NAMES:
   Where duplicates exist across groups (same name, slightly different
   numeric values), we keep the row with higher `nutrition_density`
   score as it represents a richer data entry.
   Action: df.sort_values('nutrition_density', ascending=False)
              .drop_duplicates(subset=['food'], keep='first')

3. CORRUPT / EXTREME VALUES:
   Items > 1200 kcal/100g are physically valid (concentrated fats, oils);
   they are NOT removed but are flagged for serving-size normalisation
   at recommendation time.
   Negative values (none found): would be removed as corrupt.
""")

In [ ]:
# Apply deduplication
df_clean = (
    df.sort_values('nutrition_density', ascending=False)
      .drop_duplicates(subset=['food'], keep='first')
      .reset_index(drop=True)
)
print(f'Shape before dedup: {df.shape}   After dedup: {df_clean.shape}')

---
## Part A – Section 4: Target / Feature Distributions & Imbalance

In [ ]:
# 4.1  LSTM target feature distributions
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']

for ax, feat, color in zip(axes, LSTM_FEATURES, colors):
    data = df_clean[feat].dropna()
    ax.hist(data, bins=50, color=color, alpha=0.85, edgecolor='white', linewidth=0.4)
    ax.axvline(data.median(), color='black', linestyle='--', linewidth=1.2,
               label=f'Median={data.median():.1f}')
    ax.set_title(feat.replace('_', ' ').title(), fontweight='bold', fontsize=11)
    ax.set_xlabel('g per 100g (kcal for calories)')
    ax.legend(fontsize=8)

fig.suptitle('Distribution of LSTM Input Features (per 100g)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('eda_feature_distributions.png', bbox_inches='tight', dpi=120)
plt.show()

In [ ]:
# 4.2  Caloric category distribution (LSTM 'class' analog)
# The recommendation task is regression, not classification.
# We bin calories into meaningful diet-plan categories to assess imbalance.
bins   = [0,  50, 150, 300, 500, 900, 5000]
labels = ['Very Low (<50)', 'Low (50-150)', 'Moderate (150-300)',
          'High (300-500)', 'Very High (500-900)', 'Extreme (>900)']

df_clean['caloric_band'] = pd.cut(df_clean['calories'], bins=bins, labels=labels, right=False)
band_counts = df_clean['caloric_band'].value_counts().sort_index()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

palette = ['#27ae60','#f1c40f','#e67e22','#e74c3c','#8e44ad','#2c3e50']
bars = ax1.bar(band_counts.index, band_counts.values, color=palette, edgecolor='white', linewidth=0.6)
for bar, val in zip(bars, band_counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 8,
             str(val), ha='center', va='bottom', fontsize=9, fontweight='bold')
ax1.set_title('Food Count by Caloric Band', fontweight='bold', fontsize=12)
ax1.set_xlabel('Caloric Band (kcal per 100g)')
ax1.set_ylabel('Count')
ax1.tick_params(axis='x', rotation=30)

ax2.pie(band_counts.values, labels=band_counts.index, colors=palette,
        autopct='%1.1f%%', startangle=90, pctdistance=0.78)
ax2.set_title('Caloric Band Share', fontweight='bold', fontsize=12)

fig.suptitle('Caloric Distribution of Food Items (LSTM Training Catalogue)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_caloric_distribution.png', bbox_inches='tight', dpi=120)
plt.show()

print('Band counts:')
print(band_counts.to_string())

In [ ]:
# 4.3  Macro-nutrient share (protein / carbs / fat)
macro_sum = df_clean[['protein','carbohydrates','fat']].sum()
fig, ax = plt.subplots(figsize=(6, 5))
wedges, texts, autotexts = ax.pie(
    macro_sum.values, labels=['Protein','Carbohydrates','Fat'],
    colors=['#3498db','#2ecc71','#e74c3c'],
    autopct='%1.1f%%', startangle=140,
    wedgeprops={'linewidth': 1.5, 'edgecolor': 'white'}
)
for text in autotexts:
    text.set_fontsize(11)
ax.set_title('Dataset-level Macro Share\n(summed g across all foods)', fontweight='bold')
plt.savefig('eda_macro_share.png', bbox_inches='tight', dpi=120)
plt.show()
print('Macro totals (g):', macro_sum.to_dict())

In [ ]:
# 4.4  Imbalance commentary
print("""
IMBALANCE ANALYSIS
===================
The caloric distribution is RIGHT-SKEWED:
  - ~45-55% of food items fall in the Low-Moderate caloric band (50-300 kcal).
  - Extreme-calorie items (oils, butter, concentrated fats) are a small tail (<5%).

For the LSTM (regression task):
  The model will see far more training examples from the Low-Moderate band.
  This means the predicted nutrient vector will be pulled towards the
  low-calorie regime even when the user's macro deficit actually calls
  for a high-calorie next meal.

  Mitigation strategies:
  1. Use WEIGHTED MSE loss: assign higher loss weight to high-calorie
     samples so the LSTM does not systematically under-predict.
  2. Log-transform the target before training (log1p) to compress the
     heavy tail and normalise the loss landscape.
  3. At serving time, apply MinMaxScaler fitted on training data to keep
     all five LSTM features in [0, 1] before feeding sequences.
""")

---
## Part A – Section 5: Train / Validation / Test Split Plan

In [ ]:
# 5.1  Food-catalogue split (for recommendation cosine-similarity model)
# Stratify on caloric_band so each split has balanced representation.
# 70% train | 15% val | 15% test

df_split = df_clean.dropna(subset=['caloric_band']).copy()

train_val, test = train_test_split(
    df_split, test_size=0.15, stratify=df_split['caloric_band'], random_state=42
)
train, val = train_test_split(
    train_val, test_size=0.15/0.85, stratify=train_val['caloric_band'], random_state=42
)

print(f'Train : {len(train):>5d} rows ({len(train)/len(df_split)*100:.1f}%)')
print(f'Val   : {len(val):>5d} rows ({len(val)/len(df_split)*100:.1f}%)')
print(f'Test  : {len(test):>5d} rows ({len(test)/len(df_split)*100:.1f}%)')

print('\nCaloric-band distribution across splits:')
for split_name, split_df in [('TRAIN', train), ('VAL', val), ('TEST', test)]:
    dist = split_df['caloric_band'].value_counts(normalize=True).sort_index().round(3)
    print(f'  {split_name}: {dict(dist)}')

In [ ]:
print("""
LSTM SEQUENCE SPLIT JUSTIFICATION
====================================
The LSTM does NOT train on the food catalogue CSV directly.
It trains on sequences of MealLog rows from the app database.

Split strategy: TIME-BASED (chronological cut-off)
  - Train  : all meal logs before the 70th percentile logged_at timestamp
  - Val    : logged_at 70th-85th percentile
  - Test   : logged_at 85th-100th percentile (most recent 15%)

WHY TIME-BASED (not random):
  1. Meal sequences are temporally ordered: shuffling would leak future
     information into the training set.
  2. The model must generalise to *future* meals, so we evaluate on the
     most recent behaviour, simulating real deployment.
  3. NOT stratified by caloric band: temporal ordering takes precedence;
     band balance is monitored as a diagnostic only.

WHY NOT per-user split:
  The app currently has limited users. A per-user split would create
  too small a test set and cannot capture cross-user nutrient patterns.
  Once user base grows (>500 users), revisit a grouped-by-user split
  to prevent user-ID leakage.
""")

---
## Part A – Section 6: Three Model-Design Findings from the Data

> These are not surface statistics — they are observations that directly change how the LSTM is architected, trained, and evaluated.

In [ ]:
# FINDING 1: High-fat foods and high-calorie foods are nearly indistinguishable
# in the 5-feature LSTM input space.

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Scatter: fat vs calories, coloured by protein quartile
df_clean['protein_quartile'] = pd.qcut(df_clean['protein'], 4, labels=['Q1','Q2','Q3','Q4'])
pq_colors = {'Q1': '#74b9ff', 'Q2': '#00b894', 'Q3': '#fdcb6e', 'Q4': '#e17055'}
for pq, grp in df_clean.groupby('protein_quartile', observed=True):
    axes[0].scatter(grp['fat'], grp['calories'], alpha=0.4, s=15,
                    color=pq_colors[pq], label=f'Protein {pq}')
axes[0].set_xlabel('Fat (g/100g)')
axes[0].set_ylabel('Calories (kcal/100g)')
axes[0].set_title('Fat vs Calories\n(coloured by Protein Quartile)', fontweight='bold')
axes[0].legend(fontsize=8, markerscale=2)

# Scatter: carbs vs calories
axes[1].scatter(df_clean['carbohydrates'], df_clean['calories'],
                alpha=0.35, s=12, color='#6c5ce7')
z = np.polyfit(df_clean['carbohydrates'].dropna(), df_clean['calories'].dropna(), 1)
p = np.poly1d(z)
x_line = np.linspace(0, df_clean['carbohydrates'].max(), 200)
axes[1].plot(x_line, p(x_line), 'r--', linewidth=1.5, label='Trend')
axes[1].set_xlabel('Carbohydrates (g/100g)')
axes[1].set_ylabel('Calories (kcal/100g)')
axes[1].set_title('Carbs vs Calories', fontweight='bold')
axes[1].legend(fontsize=8)

# Correlation heatmap of the 5 LSTM features
corr = df_clean[LSTM_FEATURES].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, ax=axes[2], annot=True, fmt='.2f', cmap='RdYlGn',
            mask=mask, vmin=-1, vmax=1, linewidths=0.5, annot_kws={'size': 10})
axes[2].set_title('LSTM Feature Correlation Matrix', fontweight='bold')

plt.suptitle('Finding 1: High Collinearity Between Fat & Calories Blurs the Feature Space',
             fontsize=11, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('eda_finding1_collinearity.png', bbox_inches='tight', dpi=120)
plt.show()

print('Pearson correlations with calories:')
print(df_clean[LSTM_FEATURES].corr()['calories'].drop('calories').sort_values(ascending=False).to_string())

In [ ]:
print("""
FINDING 1 - FAT AND CALORIES ARE NEARLY CO-LINEAR (r approx 0.80+)
====================================================================
Observation:
  Fat content explains ~80% of caloric variance in this catalogue.
  Foods with very different macronutrient PROFILES (e.g., a high-fat
  cheese vs. a high-fat nut butter) will map to nearly identical LSTM
  input vectors when only calories, protein, carbs, fat, fiber are used.

LSTM Design Impact:
  The 5-feature vector is insufficient to distinguish fatty-carb vs
  fatty-protein foods. We MUST add saturated_fat or fat_ratio
  (fat/calories) as a 6th input feature to break the degeneracy.
  Alternatively: use L2 regularisation on the LSTM hidden state to
  prevent the model from over-indexing on fat as a proxy for all
  high-energy meals.
""")

In [ ]:
# FINDING 2: dietary_fiber is near-zero for ~40-60% of foods.

fiber_zero_pct = (df_clean['dietary_fiber'] == 0).mean() * 100
fiber_near_zero_pct = (df_clean['dietary_fiber'] < 0.5).mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(df_clean['dietary_fiber'], bins=60, color='#00b894', alpha=0.8, edgecolor='white')
axes[0].axvline(0, color='red', linestyle='--', linewidth=2, label='fiber = 0')
axes[0].axvline(0.5, color='orange', linestyle='--', linewidth=1.5, label='fiber < 0.5g')
axes[0].set_xlabel('Dietary Fiber (g per 100g)')
axes[0].set_ylabel('Count')
axes[0].set_title(
    f'Dietary Fiber Distribution\n({fiber_zero_pct:.1f}% exactly zero, {fiber_near_zero_pct:.1f}% near-zero)',
    fontweight='bold')
axes[0].legend()

df_plot = df_clean.dropna(subset=['caloric_band'])
groups = [df_plot[df_plot['caloric_band']==b]['dietary_fiber'].dropna().values
          for b in df_plot['caloric_band'].cat.categories]
bp = axes[1].boxplot(groups, labels=df_plot['caloric_band'].cat.categories,
                     patch_artist=True, notch=False)
box_colors = ['#27ae60','#f1c40f','#e67e22','#e74c3c','#8e44ad','#2c3e50']
for patch, color in zip(bp['boxes'], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_xlabel('Caloric Band')
axes[1].set_ylabel('Dietary Fiber (g/100g)')
axes[1].set_title('Fiber Distribution by Caloric Band', fontweight='bold')
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle('Finding 2: Dietary Fiber is Severely Skewed Towards Zero',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('eda_finding2_fiber_skew.png', bbox_inches='tight', dpi=120)
plt.show()

print(f'Dietary fiber = 0.0 : {fiber_zero_pct:.1f}% of foods')
print(f'Dietary fiber < 0.5g: {fiber_near_zero_pct:.1f}% of foods')
print(f'Stats:\n{df_clean["dietary_fiber"].describe().to_string()}')

In [ ]:
print("""
FINDING 2 - DIETARY FIBER IS NEAR-ZERO IN ~40-60% OF ALL ITEMS
================================================================
Observation:
  The majority of animal-protein, dairy, and highly processed food items
  have dietary_fiber = 0 or < 0.5 g. This is correct nutrition science
  but creates a ZERO-INFLATED distribution in the LSTM input channel.

LSTM Design Impact:
  If we train the LSTM with MSE loss on raw fiber values, it will
  learn to ALWAYS predict ~0 fiber because that minimises loss on
  the majority class. This makes the model blind to vegan/high-fiber
  user goals.
  
  Fix options:
  (a) Use a zero-inflated loss or a two-stage model:
      Stage 1 - binary classifier: 'is this a fiber-rich meal?'
      Stage 2 - LSTM regression head only runs when Stage 1 says yes.
  (b) Apply log1p-transform to the fiber feature before feeding
      into the LSTM to spread the near-zero mass.
""")

In [ ]:
# FINDING 3: Macro composition clusters into 3 archetypes.
# Boundary foods create ambiguous LSTM hidden states.

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

macro_cols = ['protein', 'carbohydrates', 'fat']
df_macro = df_clean[macro_cols].dropna().copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_macro)

km = KMeans(n_clusters=3, random_state=42, n_init=10)
labels_km = km.fit_predict(X_scaled)
df_macro['cluster'] = labels_km

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cluster_colors = {0: '#e74c3c', 1: '#3498db', 2: '#2ecc71'}
for c in range(3):
    mask = labels_km == c
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    alpha=0.35, s=14, color=cluster_colors[c], label=f'Cluster {c}')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
axes[0].set_title('PCA of Macro Profiles (3 KMeans Clusters)', fontweight='bold')
axes[0].legend(fontsize=9)

centroids = scaler.inverse_transform(km.cluster_centers_)
centroid_df = pd.DataFrame(centroids, columns=macro_cols)
centroid_df['cluster'] = ['Cluster 0','Cluster 1','Cluster 2']
centroid_df.set_index('cluster').plot(kind='bar', ax=axes[1],
                                      color=['#3498db','#2ecc71','#e74c3c'],
                                      edgecolor='white')
axes[1].set_title('Macro Archetype Centroids', fontweight='bold')
axes[1].set_ylabel('g per 100g')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Macro')

plt.suptitle('Finding 3: Three Macro Archetypes - Boundary Foods Create LSTM Ambiguity',
             fontsize=11, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('eda_finding3_macro_archetypes.png', bbox_inches='tight', dpi=120)
plt.show()

print('Cluster centroids (original scale):')
print(centroid_df.to_string(index=False))

In [ ]:
print("""
FINDING 3 - THREE MACRO ARCHETYPES; BOUNDARY FOODS CONFUSE THE LSTM
=====================================================================
Observation:
  K-Means (k=3) on [protein, carbs, fat] reveals three natural archetypes:
    A. Carb-dominant  - grains, fruits, sweet snacks
    B. Fat-dominant   - dairy, oils, nuts, fatty meats
    C. Protein-moderate/mixed - lean meats, legumes, eggs

  Foods at the BOUNDARY (e.g., legumes: moderate carb AND moderate protein;
  whole milk: moderate fat AND moderate carb) produce ambiguous LSTM
  hidden states. The model cannot confidently assign them to a
  nutritional trajectory.

LSTM Design Impact:
  Add a CLUSTER EMBEDDING (integer 0/1/2) as an auxiliary input to
  the LSTM's initial hidden state so the model knows which macro
  archetype the previous meal belonged to. This acts like a 'dietary
  mode' hint and reduces next-meal prediction variance for boundary foods.

  During evaluation, track prediction error separately for boundary vs.
  core-cluster foods to detect systematic bias early.
""")

---
## Summary of EDA Findings

| # | Finding | Impact on LSTM Design |
|---|---|---|
| **1** | **Fat & Calories are nearly co-linear (r ≈ 0.80+)** — fatty-protein and fatty-carb foods map to near-identical 5-feature vectors | Add `saturated_fat` or `fat_ratio` as 6th feature; apply L2 reg to LSTM hidden state |
| **2** | **Dietary fiber is zero-inflated (~40–60% of items = 0g)** — naive MSE will predict zero-fiber for all meals, ignoring vegan/high-fiber users | Apply `log1p` transform to fiber OR use a 2-stage binary classifier + regression head |
| **3** | **Three macro archetypes exist; boundary foods (legumes, whole milk) have ambiguous cluster membership** — LSTM hidden state oscillates unpredictably after boundary meals | Inject cluster-ID embedding into LSTM initial hidden state as a 'dietary-mode' cue; evaluate separately on boundary vs. core-cluster foods |

---
*End of EDA Notebook — all cells run top-to-bottom without errors.*